In [1]:

import sys

# delete before use!!
sys.path.insert(0,"/software/local/languages/miniforge3/envs/elena/lib/python3.12/site-packages/")
sys.path.insert(0,"/user/work/yl18410/miniconda3/envs/new_graphnet_v2/lib/python3.12/site-packages")

#import cartopy
#import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import einops
import numpy as np
import torch
import os
import pickle
import random

from model.layers.encoder import *
from model.layers.decoder import *
from model.layers.processor import *
from model.layers.graph_net_block import *
from model.data.dataloader_graphnet import *
from model.data.load_data import *
#from model.data.load_data_coarsening import *
from model.forecast import GraphSatelliteForecaster, GraphSatelliteForecasterClassifier, GraphSatelliteForecasterConvClassifier
from model.loss_functions import *


import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
import time
from datetime import datetime
import json
import argparse

import random


%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import timeit

In [2]:
def baseline_mol(desired_data,months,desired_year):
    '''
    Function used to get the value for the input and output
    '''
    total_data_points = desired_data.fp_data_full.particle_locations_n.time.shape[-1]                                                                      
    # Load the CSV file
    #df = pd.read_csv('Analysis/CH4_Semihemispheric_modelled_mole_fractions.csv')
    df = pd.read_csv('/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator/CH4_Semihemispheric_modelled_mole_fractions.csv')
    baseline_list = np.zeros((total_data_points,4))

    '''
    # Specify the year and month you're interested in
    # Filter the data for the specific year and month
    filtered_data = df[(df['Year'] == specific_year) & (df['Month'] == specific_month)]

    # Extract the 4th, 5th, 6th, and 7th columns
    selected_columns = filtered_data.iloc[:, [3, 4, 5, 6]].values

    # Display the results
    print(selected_columns)
    '''

    # Iterate through the different numbers 
    #months = ['01','02','03','04','05','06','07','08','09','10','11','12']
    #year = 2016
    datetime_array = np.array(desired_data.fp_data_full.particle_locations_n.time)

    total_data_points = desired_data.fp_data_full.particle_locations_n.time.shape[-1]
    north_list = np.zeros(total_data_points) # Creates a list of size N with None values
    south_list = np.zeros(total_data_points)
    east_list = np.zeros(total_data_points)
    west_list = np.zeros(total_data_points)

    for month in months:
        # Cams field for a particular month
        if desired_year > 2017:
            # Change made due to the naming of the data
            cams = xr.open_dataset(f"/group/chemistry/acrg/LPDM/bc/SOUTHAMERICA/ch4_SOUTHAMERICA_{desired_year}{month}_CAMS-inversion_climatology.nc")
        else:
            cams = xr.open_dataset(f"/group/chemistry/acrg/LPDM/bc/SOUTHAMERICA/ch4_SOUTHAMERICA_{desired_year}{month}_CAMS-inversion.nc")    
        # Extract year and month
        #import ipdb; ipdb.set_trace()
        years = np.array([np.datetime64(date, 'Y').astype(int) + 1970 for date in datetime_array])
        months = np.array([np.datetime64(date, 'M').astype(int) % 12 + 1 for date in datetime_array])

        # Desired year and month
        #desired_year = 2016
        desired_month = month
        print('desired month',desired_month)
        
        '''
        coarse_cams  = cams.coarsen(lon=desired_data.coarsening_factor, lat=desired_data.coarsening_factor, boundary="pad").mean()
        '''
        # Find the index of the first occurrence
        indices = np.where((years == desired_year) & (months == int(desired_month)))[0]
        print(indices)
        if len(indices)>0:
            # Get the inputs
            filtered_data = df[(df['Year'] == desired_year) & (df['Month'] == int(desired_month))]
            # Extract the 4th, 5th, 6th, and 7th columns
            selected_columns = filtered_data.iloc[:, [3, 4, 5, 6]].values
            baseline_list[indices] = selected_columns/1000 # Convert from parts per trillion to parts per million

            print(indices[0])
            # Multply the first value with all the other values of the array
            print(cams.vmr_n.shape)
            # CAMS field should be stationary over the period of a month
            #import ipdb; ipdb.set_trace()

            north_mol = np.sum(cams.vmr_n * desired_data.locs.particle_locations_n[:,:,indices], axis=(0,1))
            south_mol = np.sum(cams.vmr_s * desired_data.locs.particle_locations_s[:,:,indices], axis=(0,1))
            east_mol = np.sum(cams.vmr_e * desired_data.locs.particle_locations_e[:,:,indices], axis=(0,1))
            west_mol = np.sum(cams.vmr_w * desired_data.locs.particle_locations_w[:,:,indices], axis=(0,1))
            '''
            north_mol = np.sum(coarse_cams.vmr_n * desired_data.locs.particle_locations_n[:,:,indices], axis=(0,1))
            south_mol = np.sum(coarse_cams.vmr_s * desired_data.locs.particle_locations_s[:,:,indices], axis=(0,1))
            east_mol = np.sum(coarse_cams.vmr_e * desired_data.locs.particle_locations_e[:,:,indices], axis=(0,1))
            west_mol = np.sum(coarse_cams.vmr_w * desired_data.locs.particle_locations_w[:,:,indices], axis=(0,1))
            '''
            '''
            north_mol = np.sum(cams.vmr_n * desired_data.fp_data_full.particle_locations_n[:,:,indices], axis=(0,1))
            south_mol = np.sum(cams.vmr_s * desired_data.fp_data_full.particle_locations_s[:,:,indices], axis=(0,1))
            east_mol = np.sum(cams.vmr_e * desired_data.fp_data_full.particle_locations_e[:,:,indices], axis=(0,1))
            west_mol = np.sum(cams.vmr_w * desired_data.fp_data_full.particle_locations_w[:,:,indices], axis=(0,1))
            '''
            #import ipdb; ipdb.set_trace()
            north_list[indices] = north_mol
            south_list[indices] = south_mol
            east_list[indices] = east_mol
            west_list[indices] = west_mol


        print(baseline_list)
        #cams = xr.open_dataset("/group/chemistry/acrg/LPDM/bc/SOUTHAMERICA/ch4_SOUTHAMERICA_201611_CAMS-inversion.nc")
        # Making the assumption that the values in the month are not different, get the first value
        # Multiple the different values
    
    return baseline_list, north_list, south_list, east_list, west_list

In [3]:

#### 1 Set up


parameters = {
    "model_name" : "test_run",
    "train_load_data" : {
        "year":"2013",
        "freq":100,
        "region":"BRAZIL",
        "size":50,
        "verbose":True,
        "met_args":{"met_levels":[3, 15,21]},
        "load_everything":True,
    },

    "test_load_data" : {
        "year":"2017",
        "freq":300
    },

    "variables" : {
        #"met_variables":{"x_wind":[3,15], "y_wind":[3,15], "upward_air_velocity":[3,15], "atmosphere_boundary_layer_thickness":[], "surface_air_pressure":[]},
        "met_variables":{"x_wind":[3,15], "y_wind":[3,15],"surface_air_pressure":[]},
        "static_variables":["lat_coords", "lon_coords", "x_coords", "y_coords", "topog", "landcover"],
        "time_deltas":[6,12]
    },

    "dataloader_parameters":{
        "input_transforms":["clever_transform_3"],
        "output_transforms":["logv4"]  
    },

    "model_parameters":{
        "num_blocks":4, 
        "node_dim":64, 
        "edge_dim":64, 
        "hidden_layers_processor_node":2, 
        "hidden_layers_processor_edge":2,  
        "hidden_layers_decoder":1, 
        "hidden_dim_processor_node":16, 
        "hidden_dim_processor_edge":16, 
        "hidden_dim_decoder":16, 
        "resolution":4, 
        "output_dim":1, 
        "residuals":False, 
        "attention":False
    },

    "learning_rate":5e-5,

    "loss_functions" : {
        "criterion": "torch.nn.MSELoss()",
        "criterion_test": "torch.nn.MSELoss()"
    }    
}






In [4]:

parameters = {
    "model_name" : "test_run",
    "train_load_data" : {
        "year":"2013",
        "freq":100,
        "region":"BRAZIL",
        #"domain_to_cut":{"lat":[-24.3,20]},
        #"domain_to_cut":{"lat":[-39.7,4.7]}, # Lat 190, lon 190
        #"domain_to_cut":{"lat":[-35.5,-0.5], "lon":[-77.5,7.5]}, # Lat 150,lon 150
        "domain_to_cut":{"lat":[-29.5,-6], "lon":[-60,-10]}, # Lat 100,lon 100
        #"domain_to_cut":{"lat":[-23.7,-12], "lon":[-43.5,-26]}, # Lat 50,lon 50
        #"coarsening_factor":2,
        "verbose":True,
        "met_args":{"met_levels":[3, 15,21]},
    },

    "test_load_data" : {
        "year":"2016",
        "freq":300
    },

    "variables" : {
        #"met_variables":{"x_wind":[3,15], "y_wind":[3,15], "upward_air_velocity":[3,15], "atmosphere_boundary_layer_thickness":[], "surface_air_pressure":[]},
        "met_variables":{"x_wind":[3,15], "y_wind":[3,15],"surface_air_pressure":[]},
        "time_deltas":[0]
    },

    "dataloader_parameters":{
        "input_transforms":["clever_transform_3"],
    },

    "model_parameters":{
        "num_blocks":4, 
        "node_dim":64, 
        "edge_dim":64, 
        "hidden_layers_processor_node":2, 
        "hidden_layers_processor_edge":2,  
        "hidden_layers_decoder":1, 
        "hidden_dim_processor_node":16, 
        "hidden_dim_processor_edge":16, 
        "hidden_dim_decoder":16, 
        "resolution":4, 
        "output_dim":1, 
        "residuals":False, 
        "attention":False
    },

    "learning_rate":5e-7,

    "loss_functions" : {
        "criterion": "torch.nn.MSELoss()",
        "criterion_test": "torch.nn.MSELoss()"
    }    
}




parameters2 = {
    "model_name" : "test_run",
    "train_load_data" : {
        "year":"2015",
        "freq":100,
        "region":"BRAZIL",
        "verbose":True,
        "coarsening_factor":1,
    },

    "test_load_data" : {
        "year":"2016",
        "freq":300
    },

    "variables" : {
        "met_variables":{"x_wind":[3,15], "y_wind":[3,15], "upward_air_velocity":[3,15],"atmosphere_boundary_layer_thickness":[], "surface_air_pressure":[]},
        #"static_variables":["sin_lat_coords", "sin_lon_coords", "cos_lat_coords", "cos_lon_coords", "lat_coords", "lon_coords", "x_coords", "y_coords", "topog"],
        #"met_variables":{"x_wind":[3,9,15,21,30,42,51], "wind_speed":[3,30,51], "wind_angle":[3,30,51], "y_wind":[3,9,15,21,30,42,51], "upward_air_velocity":[3,9,15,21,30,42,51], "air_temperature":[3,9,15,21,30,42,51], "air_pressure":[3,9,15,21,30,42,51], "atmosphere_boundary_layer_thickness":[], "surface_air_pressure":[]},
        #"static_variables":["sin_lat_coords", "sin_lon_coords", "cos_lat_coords", "cos_lon_coords", "lat_coords", "lon_coords", "x_coords", "y_coords", "topog","domain_binary_release", "domain_distance_release"],
        "time_deltas":[0],
    },

    "dataloader_parameters":{
        "input_transforms":["clever_transform_3"],
    },

    "model_parameters":{
        "num_blocks":4, 
        "node_dim":64, 
        "edge_dim":64, 
        "hidden_layers_processor_node":2, 
        "hidden_layers_processor_edge":2,  
        "hidden_layers_decoder":1, 
        "hidden_dim_processor_node":16, 
        "hidden_dim_processor_edge":16, 
        "hidden_dim_decoder":16, 
        "resolution":4, 
        "output_dim":8, 
        "residuals":False, 
        "attention":False
    },
    'normalization':'all',
    "learning_rate":5e-7,
    "seed":42,
    "epochs":2,
    "num_classes":4,
    "use_baselines":False,

    "loss_functions" : {
        "criterion": "torch.nn.MSELoss()",
        "criterion_test": "torch.nn.MSELoss()"
    }    
}



In [5]:

print("PARAMETERS:")
print(parameters)

model_name = parameters["model_name"]
print(model_name)


if "seed" in (parameters.keys()):
    print(parameters["seed"])
    np.random.seed(parameters["seed"])
    torch.manual_seed(parameters["seed"])
    torch.cuda.manual_seed(parameters["seed"])
    random.seed(parameters["seed"])

else:
    print("34")
    np.random.seed(34)
    torch.manual_seed(34)
    torch.cuda.manual_seed(34)
    random.seed(34)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### 2 Load Data


train_load_data = copy.deepcopy(parameters["train_load_data"])
# load train parameters and upload with any changes to test data
test_load_data = copy.deepcopy(parameters["train_load_data"])
test_load_data.update(parameters["test_load_data"])
print(train_load_data)
print(test_load_data)

data = LoadDomainSatelliteData(**train_load_data)
test_data = LoadDomainSatelliteData(**test_load_data)

PARAMETERS:
{'model_name': 'test_run', 'train_load_data': {'year': '2013', 'freq': 100, 'region': 'BRAZIL', 'domain_to_cut': {'lat': [-29.5, -6], 'lon': [-60, -10]}, 'verbose': True, 'met_args': {'met_levels': [3, 15, 21]}}, 'test_load_data': {'year': '2016', 'freq': 300}, 'variables': {'met_variables': {'x_wind': [3, 15], 'y_wind': [3, 15], 'surface_air_pressure': []}, 'time_deltas': [0]}, 'dataloader_parameters': {'input_transforms': ['clever_transform_3']}, 'model_parameters': {'num_blocks': 4, 'node_dim': 64, 'edge_dim': 64, 'hidden_layers_processor_node': 2, 'hidden_layers_processor_edge': 2, 'hidden_layers_decoder': 1, 'hidden_dim_processor_node': 16, 'hidden_dim_processor_edge': 16, 'hidden_dim_decoder': 16, 'resolution': 4, 'output_dim': 1, 'residuals': False, 'attention': False}, 'learning_rate': 5e-07, 'loss_functions': {'criterion': 'torch.nn.MSELoss()', 'criterion_test': 'torch.nn.MSELoss()'}}
test_run
34
{'year': '2013', 'freq': 100, 'region': 'BRAZIL', 'domain_to_cut': {'

: 

In [ ]:
data.met

In [ ]:
data.met

In [ ]:
train_months = ['01','02','03','04','05','06','07','08','09','10','11','12']
# TODO: Nawid- get the train year and the test year from the trainload data 
train_year = 2015
    
test_months = ['01','02','03','04','05','06','07','08','09','10','11','12']
test_year = 2016

baseline_list, north_list, south_list, east_list, west_list = baseline_mol(data,train_months, train_year)

test_baseline_list, test_north_list, test_south_list, test_east_list, test_west_list = baseline_mol(test_data,test_months, test_year)

outputs = np.stack((north_list,south_list, east_list,west_list),axis=1)
print('outputs',south_list)
test_outputs = np.stack((test_north_list,test_south_list, test_east_list,test_west_list),axis=1)
normalization = 'separate'
if normalization =='separate':
    outputs_mean_values, outputs_std_values = np.mean(outputs,axis=0), np.std(outputs,axis=0)
    baseline_mean_values, baseline_std_values = np.mean(baseline_list,axis=0), np.std(baseline_list,axis=0)
    outputs =  (outputs-outputs_mean_values)/outputs_std_values        
    baseline_list =  (baseline_list-baseline_mean_values)/baseline_std_values

    test_outputs = (test_outputs-outputs_mean_values)/outputs_std_values
    test_baseline_list = (test_baseline_list-baseline_mean_values)/baseline_std_values

elif parameters['normalization'] =='all':
    outputs_mean_values, outputs_std_values = np.mean(outputs), np.std(outputs)
    baseline_mean_values, baseline_std_values = np.mean(baseline_list), np.std(baseline_list)
    outputs =  (outputs-outputs_mean_values)/outputs_std_values
    baseline_list =  (baseline_list-baseline_mean_values)/baseline_std_values

    test_outputs = (test_outputs-outputs_mean_values)/outputs_std_values
    test_baseline_list = (test_baseline_list-baseline_mean_values)/baseline_std_values

has_nan = np.isnan(outputs).any()
print("Contains NaNs:", has_nan)

has_nan = np.isnan(test_outputs).any()
print("Contains NaNs:", has_nan)


In [ ]:
denormalized_outputs = (outputs*outputs_std_values)+ outputs_mean_values
# Step 2: Convert to float32
outputs_float32 = outputs.astype(np.float32)

# Step 3: Convert back to float64
outputs_float64 = outputs_float32.astype(np.float64)

# Step 4: Denormalize (assuming you have a denormalization function)
# For example, if you normalized like this: (x - mean) / std
# then you denormalize like this:

# Denormalize both original and reconverted

denorm_reconverted = (outputs_float64 * outputs_std_values) + outputs_mean_values

# Step 5: Calculate error (e.g., Mean Absolute Error or Mean Squared Error)
mae = np.mean(np.abs(denormalized_outputs - denorm_reconverted))
mse = np.mean((denormalized_outputs - denorm_reconverted) ** 2)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)

In [ ]:
print(south_list)

In [ ]:
data.domain_size


In [ ]:
input_variables = parameters["variables"]

inputs, names = get_square_satellite_inputs(data, **input_variables, return_variable_names=True, return_asarray=False)
#test_inputs = get_square_satellite_inputs(test_data, **input_variables, return_asarray=True)

In [ ]:
inputs

In [ ]:
def clever_transform_3_to_dataset(
    inputs,
    mode="train",
    saved_stats=None,
    ignore_vars=("sin_time_year", "cos_time_year"),
    landcover_keywords=("land_cover", "landcover"),
):
    """
    Apply standard or min-max scaling to a Dask-backed xarray.DataArray with a MultiIndex variable_name.

    Parameters
    ----------
    inputs : xarray.DataArray
        Dimensions: (fp_time, lat, lon, variable_name), where variable_name is a MultiIndex (varname, level, stash)
    mode : str
        "train" or "test"
    saved_stats : dict or None
        If test mode, supply saved mean/std or min/max values
    ignore_vars : tuple of str
        Variable names to leave untransformed
    landcover_keywords : tuple of str
        If varname contains one of these, use MinMaxScaler instead of StandardScaler

    Returns
    -------
    dataset : xarray.Dataset
        Transformed variables as separate DataArrays named {varname}_L{level}
    stats : dict
        Dictionary of scalers/statistics (only in train mode)
    """

    assert "variable_name" in inputs.dims
    var_index = inputs.coords["variable_name"].to_index()
    stats = {} if mode == "train" else saved_stats
    print('var indices',var_index)
    output_vars = {}

    for i, (varname, level, stash) in enumerate(var_index):
        print('i:',i)
        print('varname:',varname)
        if varname in ignore_vars:
            continue

        da = inputs.isel(variable_name=i)
        print('obtained data')
        # Name of the variable in the output Dataset
        out_name = f"{varname}_L{level}"

        # Determine scaling type
        use_minmax = any(keyword in varname for keyword in landcover_keywords)

        reduce_dims = ('fp_time', 'lat', 'lon')

        if mode == "train":
            if use_minmax:
                vmin = da.min(dim=reduce_dims, skipna=True)
                vmax = da.max(dim=reduce_dims, skipna=True)
                stats[(varname, level)] = {"min": float(vmin.compute()), "max": float(vmax.compute())}
                scaled = (da - vmin) / (vmax - vmin + 1e-6)
            else:
                mean = da.mean(dim=reduce_dims, skipna=True)
                std = da.std(dim=reduce_dims, skipna=True)
                stats[(varname, level)] = {"mean": float(mean.compute()), "std": float(std.compute())}
                scaled = (da - mean) / (std + 1e-6)

        elif mode == "test":
            if use_minmax:
                vmin = stats[(varname, level)]["min"]
                vmax = stats[(varname, level)]["max"]
                scaled = (da - vmin) / (vmax - vmin + 1e-6)
            else:
                mean = stats[(varname, level)]["mean"]
                std = stats[(varname, level)]["std"]
                scaled = (da - mean) / (std + 1e-6)

        output_vars[out_name] = scaled

    # Combine into a new Dataset
    dataset = xr.merge(
            [da.rename(varname) for varname, da in output_vars.items()],
            compat="override"
        )
    '''
    dataset = xr.Dataset(output_vars,compat="override")
    '''
    return (dataset, stats) if mode == "train" else dataset

In [ ]:
from model.data.load_data import BoundaryDatasetXR
use_baselines = False
train_dataset = BoundaryDatasetXR(
    inputs, 
    outputs,
    input_transforms=["clever_transform_3"], 
    input_names=names
)
#train_dataset = BoundaryDatasetXR(inputs,baseline_list,outputs,use_baselines=use_baselines,input_names=names, **parameters["dataloader_parameters"])
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)

In [ ]:
transformed_ds, transform_stats = clever_transform_3_to_dataset(inputs, mode="train")

In [ ]:
)

In [ ]:
subset = inputs.isel(fp_time= 

In [ ]:
subset.shape

In [ ]:
inputs.load()

In [ ]:
# the model gets built with respect to a "reference footprint", and all predictions are done on this grid. An improvement would be to explore a way to select the best reference footrpint, or to find a way to do this dynamically for each footprint
grid, _ = get_grid(data, parameters.get("grid_reference_fp"))
print("setting up model")

test_batch_size=10


In [ ]:
data.met

In [ ]:
has_nan = np.isnan(outputs).any()

print("Contains NaNs:", has_nan)

In [ ]:
use_baselines = False
train_dataset = BoundaryDataset(inputs,baseline_list,outputs,use_baselines=use_baselines,input_names=names, **parameters["dataloader_parameters"])
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
'''
test_dataset = BoundaryDataset(test_inputs,test_baseline_list,test_outputs,use_baselines= use_baselines,input_names=names, **parameters["dataloader_parameters"])
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)
'''
'''
train_dataset =FootprintsDatasetV3(inputs, data.fp_data, input_names=names,size=data.domain_size,**parameters["dataloader_parameters"])
print(train_dataset.transform_parameters)
test_dataset = FootprintsDatasetV3(test_inputs, test_data.fp_data, input_names=names,size=test_data.domain_size, test_mode=train_dataset.transform_parameters, **parameters["dataloader_parameters"])
'''

In [ ]:
has_nan = np.isnan(inputs).any()
print("Contains NaNs:", has_nan)

#has_nan = np.isnan(test_inputs).any()
#print("Contains NaNs:", has_nan)

In [ ]:
with open('grid.pkl', 'rb') as f:
    loaded_grid = pickle.load(f)

aux_dim = len(input_variables["static_variables"]) 
feature_dim=np.shape(inputs)[-1]-aux_dim
num_classes =4
# Should probably update the name!!

model = GraphSatelliteForecasterClassifier(grid, whole_world=False, feature_dim=24, aux_dim=0,num_classes = num_classes, **parameters["model_parameters"])


#model = GraphSatelliteForecasterClassifier(loaded_grid, whole_world=False, feature_dim=24, aux_dim=0,num_classes = num_classes, **parameters["model_parameters"])
#model = GraphSatelliteForecaster(loaded_grid, whole_world=False, feature_dim=24, aux_dim=0, **parameters["model_parameters"])


In [ ]:
# transform data - 
'''
train_dataset = FootprintsDatasetV3(inputs, data.fp_data, input_names=names, **parameters["dataloader_parameters"])
test_dataset = FootprintsDatasetV3(test_inputs, test_data.fp_data, input_names=names, test_mode=train_dataset.transform_parameters, **parameters["dataloader_parameters"])
'''

# all the necessary data is already in the loaders, so we can delete the objects
#del data, test_data

lr = parameters["learning_rate"]
print(lr)
#aux_dim = len(input_variables["static_variables"]) 
#feature_dim=np.shape(inputs)[-1]-aux_dim
feature_dim = np.shape(inputs)[-1]
aux_dim=0
num_classes =4
# Should probably update the name!!

model = GraphSatelliteForecasterClassifier(grid, whole_world=False, feature_dim=feature_dim, aux_dim=aux_dim,num_classes = num_classes, **parameters["model_parameters"])

'''
num_lat, num_lon = len(data.met.lat.values), len(data.met.lon.values)
model = GraphSatelliteForecasterConvClassifier(grid, whole_world=False, feature_dim=feature_dim, aux_dim=0,num_classes = num_classes,input_height = num_lat, input_width=num_lon, **parameters["model_parameters"])
'''
#model = GraphSatelliteForecaster(grid, whole_world=False, feature_dim=feature_dim, aux_dim=aux_dim, **parameters["model_parameters"])

#### 3 Make model

# this is leftover from the previous model and actually shouldnt make a difference

In [ ]:
feature_dim=np.shape(inputs)[-1]
print(feature_dim)
print(aux_dim)

In [ ]:
inputs.shape

In [ ]:

criterion = eval(parameters["loss_functions"]["criterion"])

criterion_test = eval(parameters["loss_functions"]["criterion_test"])

lr = parameters["learning_rate"]
optimizer = optim.AdamW(model.parameters(), lr=lr)

losses = {"train":[], "test":[], "NMAE_test":[], "MSE_test_transformed":[], "NMAE_test_transformed":[], "accuracy":[], "IoU":[]}

NMAE_function = NMAE


#### 3 Dump info
print("saving grids etc")

epoch_so_far = 0
if torch.cuda.is_available():
    model.cuda()

#### 4 Train loop
for epoch in range(302):
    epoch=epoch+epoch_so_far
    running_loss = 0.0
    print(f"Start Epoch: {epoch}")
    start = time.time()
    for i, batch in enumerate(train_loader):
        # get the inputs; data is a list of [inputs, labels]
        ins, labels = batch[0].to(device), batch[1].to(device)
        
        #has_nan = np.isnan(ins).any()

        #print("Contains NaNs:", has_nan)
        
        optimizer.zero_grad()
        # forward + backward + optimize
        model_outputs = model(ins)
        #print('ins',ins)
        #print('labels',labels)
        loss = criterion(model_outputs, labels)
        loss.backward()
        optimizer.step()
        loss = criterion_test(model_outputs, labels)
        running_loss += loss.item()
        '''
        inputs_has_nan = np.isnan(ins.detach().numpy()).any()
        print("Contains NaNs:", inputs_has_nan)

        labels_has_nan = np.isnan(labels.detach().numpy()).any()
        print("Contains NaNs:", labels_has_nan)

        has_nan = np.isnan(model_outputs.detach().numpy()).any()
        print(model_outputs)

        print("Contains NaNs:", has_nan)
        '''
        end = time.time()
        del model_outputs 
        if i % 10==0:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.3f} Time: {end - start} sec")
    '''
    test_error = 0.0
    test_out = np.zeros((test_dataset.inputs.size()[0], test_dataset.inputs.size()[1]))   
    for i_test, batch in enumerate(test_loader):
        # get the inputs; data is a list of [inputs, labels]
        ins, labels = batch[0].to(device), batch[1].to(device)
        test_error += criterion_test(model(ins), labels).item()
        test_out[i_test*test_batch_size:(i_test+1)*test_batch_size,:] = np.squeeze(model(ins).detach().cpu().numpy())
    '''
    
    losses["train"].append(running_loss/(i+1))
    '''
    losses["test"].append(test_error/(i_test+1))
    '''
    '''
    #evaluate
    truths = torch.squeeze(test_dataset.fp).detach().numpy()
    print(f"NMAE: {NMAE(test_out,truths)}")
    losses["NMAE_test"].append(NMAE(test_out,truths))
    transformed_preds = test_dataset.inverse_transform(test_out)
    evaluation_metrics = test_dataset.evaluate()
    losses["NMAE_test_transformed"].append(evaluation_metrics["NMAE"])
    losses["MSE_test_transformed"].append(evaluation_metrics["MSE"])
    losses["accuracy"].append(evaluation_metrics["Accuracy"])
    losses["IoU"].append(evaluation_metrics["IOU"])
    '''


### Square case - modelling

In [ ]:
parameters = {
    "model_name" : "test_run",
    "train_load_data" : {
        "year":"2015",
        "freq":100,
        "region":"BRAZIL",
        "size":180,
        "load_everything":True,
        "coarsening_factor":2,
        "verbose":True,
        "met_args":{"met_levels":[3, 15,21]},
    },

    "test_load_data" : {
        "year":"2016",
        "freq":300
    },

    "variables" : {
        "met_variables":{"x_wind":[3,15], "y_wind":[3,15], "upward_air_velocity":[3,15], "atmosphere_boundary_layer_thickness":[], "surface_air_pressure":[]},
        "time_deltas":[6,12]
    },

    "dataloader_parameters":{
        "input_transforms":["clever_transform_3"],
    },

    "model_parameters":{
        "num_blocks":4, 
        "node_dim":64, 
        "edge_dim":64, 
        "hidden_layers_processor_node":2, 
        "hidden_layers_processor_edge":2,  
        "hidden_layers_decoder":1, 
        "hidden_dim_processor_node":16, 
        "hidden_dim_processor_edge":16, 
        "hidden_dim_decoder":16, 
        "resolution":4, 
        "output_dim":8, 
        "residuals":False, 
        "attention":False
    },

    "learning_rate":5e-7,

    "loss_functions" : {
        "criterion": "torch.nn.MSELoss()",
        "criterion_test": "torch.nn.MSELoss()"
    }   
} 

In [ ]:

print("PARAMETERS:")
print(parameters)

model_name = parameters["model_name"]
print(model_name)


if "seed" in (parameters.keys()):
    print(parameters["seed"])
    np.random.seed(parameters["seed"])
    torch.manual_seed(parameters["seed"])
    torch.cuda.manual_seed(parameters["seed"])
    random.seed(parameters["seed"])

else:
    print("34")
    np.random.seed(34)
    torch.manual_seed(34)
    torch.cuda.manual_seed(34)
    random.seed(34)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### 2 Load Data


train_load_data = copy.deepcopy(parameters["train_load_data"])
# load train parameters and upload with any changes to test data
test_load_data = copy.deepcopy(parameters["train_load_data"])
test_load_data.update(parameters["test_load_data"])
print(train_load_data)
print(test_load_data)

if "size" in (parameters["train_load_data"].keys()):
    print('square_domain')
    data = LoadSquareSatelliteData(**train_load_data)
    test_data = LoadSquareSatelliteData(**test_load_data)
    
else:    
    print('fixed_domain')
    data = LoadDomainSatelliteData(**train_load_data)
    test_data = LoadDomainSatelliteData(**test_load_data)

In [ ]:
data.met

In [ ]:
train_months = ['01','02','03','04','05','06','07','08','09','10','11','12']
# TODO: Nawid- get the train year and the test year from the trainload data 
train_year = 2015
    
test_months = ['01','02','03','04','05','06','07','08','09','10','11','12']
test_year = 2016

baseline_list, north_list, south_list, east_list, west_list = baseline_mol(data,train_months, train_year)
test_baseline_list, test_north_list, test_south_list, test_east_list, test_west_list = baseline_mol(test_data,test_months, test_year)

outputs = np.stack((north_list,south_list, east_list,west_list),axis=1)
print('outputs',south_list)
test_outputs = np.stack((test_north_list,test_south_list, test_east_list,test_west_list),axis=1)
normalization = 'separate'
if normalization =='separate':
    outputs_mean_values, outputs_std_values = np.mean(outputs,axis=0), np.std(outputs,axis=0)
    baseline_mean_values, baseline_std_values = np.mean(baseline_list,axis=0), np.std(baseline_list,axis=0)
    outputs =  (outputs-outputs_mean_values)/outputs_std_values        
    baseline_list =  (baseline_list-baseline_mean_values)/baseline_std_values

    test_outputs = (test_outputs-outputs_mean_values)/outputs_std_values
    test_baseline_list = (test_baseline_list-baseline_mean_values)/baseline_std_values

elif parameters['normalization'] =='all':
    outputs_mean_values, outputs_std_values = np.mean(outputs), np.std(outputs)
    baseline_mean_values, baseline_std_values = np.mean(baseline_list), np.std(baseline_list)
    outputs =  (outputs-outputs_mean_values)/outputs_std_values
    baseline_list =  (baseline_list-baseline_mean_values)/baseline_std_values

    test_outputs = (test_outputs-outputs_mean_values)/outputs_std_values
    test_baseline_list = (test_baseline_list-baseline_mean_values)/baseline_std_values

has_nan = np.isnan(outputs).any()
print("Contains NaNs:", has_nan)

has_nan = np.isnan(test_outputs).any()
print("Contains NaNs:", has_nan)


In [ ]:
input_variables = parameters["variables"]
inputs, names = get_square_satellite_inputs(data, **input_variables, return_variable_names=True, return_asarray=True)
#test_inputs = get_square_satellite_inputs(test_data, **input_variables, return_asarray=True)

In [ ]:
test_inputs = get_square_satellite_inputs(test_data, **input_variables, return_asarray=True)

In [ ]:
test_inputs_has_nan = np.isnan(test_inputs).any()
print("Contains NaNs:", test_inputs_has_nan)


In [ ]:
use_baselines = False
train_dataset = BoundaryDataset(inputs,baseline_list,outputs,use_baselines=use_baselines,input_names=names, **parameters["dataloader_parameters"])
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_dataset = BoundaryDataset(test_inputs,test_baseline_list,test_outputs,use_baselines= use_baselines,input_names=names, **parameters["dataloader_parameters"])
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

In [ ]:
# the model gets built with respect to a "reference footprint", and all predictions are done on this grid. An improvement would be to explore a way to select the best reference footrpint, or to find a way to do this dynamically for each footprint
grid, _ = get_grid(data, parameters.get("grid_reference_fp"))
print("setting up model")

test_batch_size=10

In [ ]:
len(grid)

In [ ]:
lr = parameters["learning_rate"]
print(lr)
#aux_dim = len(input_variables["static_variables"]) 
#feature_dim=np.shape(inputs)[-1]-aux_dim
feature_dim = np.shape(inputs)[-1]
aux_dim=0
num_classes =4
# Should probably update the name!!
model = GraphSatelliteForecasterClassifier(grid, whole_world=False, feature_dim=feature_dim, aux_dim=aux_dim,num_classes = num_classes, **parameters["model_parameters"])


In [ ]:
criterion = eval(parameters["loss_functions"]["criterion"])

criterion_test = eval(parameters["loss_functions"]["criterion_test"])

lr = parameters["learning_rate"]
optimizer = optim.AdamW(model.parameters(), lr=lr)

losses = {"train":[], "test":[], "NMAE_test":[], "MSE_test_transformed":[], "NMAE_test_transformed":[], "accuracy":[], "IoU":[]}

NMAE_function = NMAE


#### 3 Dump info
print("saving grids etc")

epoch_so_far = 0
if torch.cuda.is_available():
    model.cuda()

#### 4 Train loop
for epoch in range(302):
    epoch=epoch+epoch_so_far
    running_loss = 0.0
    print(f"Start Epoch: {epoch}")
    start = time.time()
    for i, batch in enumerate(train_loader):
        # get the inputs; data is a list of [inputs, labels]
        ins, labels = batch[0].to(device), batch[1].to(device)
        
        #has_nan = np.isnan(ins).any()

        #print("Contains NaNs:", has_nan)
        
        optimizer.zero_grad()
        # forward + backward + optimize
        #print(ins.shape)
        model_outputs = model(ins)
        #print('ins',ins)
        #print('labels',labels)
        loss = criterion(model_outputs, labels)
        loss.backward()
        optimizer.step()
        loss = criterion_test(model_outputs, labels)
        running_loss += loss.item()
        '''
        inputs_has_nan = np.isnan(ins.detach().numpy()).any()
        print("Contains NaNs:", inputs_has_nan)

        labels_has_nan = np.isnan(labels.detach().numpy()).any()
        print("Contains NaNs:", labels_has_nan)

        has_nan = np.isnan(model_outputs.detach().numpy()).any()
        print(model_outputs)

        print("Contains NaNs:", has_nan)
        '''
        end = time.time()
        del model_outputs 
        if i % 10==0:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.3f} Time: {end - start} sec")
    
    test_error = 0.0
    test_out = np.zeros((test_dataset.inputs.size()[0], test_dataset.inputs.size()[1]))   
    for i_test, batch in enumerate(test_loader):
        # get the inputs; data is a list of [inputs, labels]
        ins, labels = batch[0].to(device), batch[1].to(device)
        test_error += criterion_test(model(ins), labels).item()
        test_out[i_test*test_batch_size:(i_test+1)*test_batch_size,:] = np.squeeze(model(ins).detach().cpu().numpy())
    
    
    losses["train"].append(running_loss/(i+1))
    losses["test"].append(test_error/(i_test+1))

### Square class

the square class cuts the data to a square of size size x size around the measurement coordinates, with the meteorology and topography also aligned to this format

parameters:
- year, month
- region (domain is passed automatically if region is known)
- met_args: dict of shape {"met_levels":list, "met_variables":list} - subsets the meteorology to the passed levels and variables. useful to reduce loading time and memory
- subsetting the data in time: 
    - freq: the final dataset will have approx total_n_footprints/freq samples
    - sampling_mode: "regular" (sampling regularly in time, eg 1 in every freq footprints) or "random" (subsamples n_footprints/freq, time indeces, chosen at random) (default is regular)
- lazy_load: wether to load arrays onto memory, or only lazy-load them. try loading the data then getting the inputs (see below) with and without lazy_load!

In [ ]:
cropped_data = LoadSquareSatelliteData(year=2016, month ='01', region="SAHARA", met_args = {"met_levels":[3, 15]}, freq=20,  size=50, verbose=True, load_everything=True, lazy_load=False)

In [ ]:
cropped_data.met_file

In [ ]:
cropped_data.met # met object, interpolated to the same time as the footprints and cropped to the right size

# Setting up the data

The variables to extract should be passed as follows:
- met_variables: a dict of meteorological variables, of format {"var_name":[list of levels], "surface_var_name":[]}
- static_variables: a list of static variables to add. see `get_static_variables_functions` for a list of valid parameters, or add your own

other parameters:
- time_deltas (list of ints): met is extracted for eahc t-H hours, where H is each item in te list. t=0 (ie the time of the satellite measurement) is extracted automatically. time_deltas=[6,12] extracts the data at t=0, t-6h and t-12h.
- return_asarray (bool, default=False): if false, returns as an xarray dataset. if true, loads into memory as a np array of shape (time, lat*lon, variables)


In [ ]:
variables = {"x_wind":[3,15], "y_wind":[3], "surface_air_pressure":[]}
static_variables=["lat_coords", "lon_coords" , "x_coords", "y_coords", "topog", "landcover"]
inputs, input_names = get_square_satellite_inputs(cropped_data, variables, static_variables=static_variables, time_deltas=[24], return_variable_names=True, return_asarray=False)

In [ ]:
inputs # the inputs are a stacked DataArray, of dimensions fp_time, lat, lon, and variable
# the inputs are lazy loaded, so they're not actually in memory until you run inputs.load(), 
# or until you attempt to use or extract the values

In [ ]:
inputs.variable_name # contains tuples specific to each 2D variable, specifying (variable, level, time_delta)
# surface variables always have level=0, and static variables have level=0 and time_delta=0

In [ ]:
input_names # input_names is a dict with the same vaues as inputs.variable_name, but it's easier to search

we can use the xarray dataset to inspect and plot the variable more easily

In [ ]:
d = inputs.fp_time[1]
plt.imshow(inputs.sel(variable_name=('y_wind', 3, 0), fp_time=d), origin="lower")

In [ ]:
d = inputs.fp_time[1]
plt.imshow(inputs.sel(variable_name=('topog', 0, 0), fp_time=d), origin="lower")

In [ ]:
variables = {"x_wind":[3,15], "y_wind":[3], "surface_air_pressure":[]}
static_variables=["lat_coords", "lon_coords", "x_coords", "y_coords", "topog", "landcover"]
inputs_array, input_names = get_square_satellite_inputs(cropped_data, variables, static_variables=static_variables, time_deltas=[24], return_variable_names=True, return_asarray=True)
# passing return_asarray=True loads the inputs into memory and returns them with the shape (time, lat*lon, variables). might take a while!

In [ ]:
# passing return_asarray=True loads the inputs into memory and returns them with the shape (time, lat*lon, variables)
# this is what the transform function is expecting, but could be improved or avoided!
np.shape(inputs_array)

### Preparing the dataset
FootprintsDataset applies any necessary transforms on the data. the functions used to transform the data are saved in the dataset object. they can be passed to another instance FootprintsDataset (using `test_mode=train_dataset.transform_parameters`), to apply the known transforms to an unseen dataset

In [ ]:
dataset = FootprintsDataset(inputs=inputs, fp=cropped_data.fp_data, input_names=input_names, input_transforms=["clever_transform_3"], output_transforms= ["logv4"])

# Fixed domain data

LoadDomainSatelliteData will load the data over a fixed domain in space, rather than centered around the release point. if we dont specify a domain, it will just load the maximum possible domain.
You can otherwise specify the domain to cut, in latitude and longitude with domain_to_cut (see below)

In [ ]:
met_args = {"met_levels":[3, 15]}
data_dom = LoadDomainSatelliteData(year="2016", region="BRAZIL", month="06", freq=40, verbose=True, met_args=met_args)#, domain_to_cut={"lat":[0,20], "lon":[20,40]})

In [ ]:
plt.imshow(data_dom.topog.topog, origin="lower")

In [ ]:
met_args = {"met_levels":[3, 15]}
data_dom = LoadDomainSatelliteData(year="2016", region="BRAZIL", month="06", freq=40, verbose=True, met_args=met_args, domain_to_cut={"lat":[-20,20]})

In [ ]:
plt.imshow(data_dom.topog.topog, origin="lower")

In [ ]:
# the input function works with square data too. note that most of the static variable functions wont work! need to write specific ones
inputs, names = get_square_satellite_inputs(data_dom, {"x_wind":[3,15], "y_wind":[3], "surface_air_pressure":[]}, static_variables=["lat_coords", "topog", "landcover", "landcover_disaggregated"], time_deltas=[100], verbose=True, return_asarray=False, return_variable_names=True)

In [ ]:
inputs.shape # Shape (time, lat, lon, features)

In [ ]:
grid,_  = get_grid(data_dom)

In [ ]:
dataset = FootprintsDataset(inputs=inputs, fp=cropped_data.fp_data, input_names=input_names, input_transforms=["clever_transform_3"], output_transforms= ["logv4"])

In [ ]:
train_loader = DataLoader(dataset, batch_size=5, shuffle=True)

In [ ]:
model = GraphSatelliteForecaster(grid, whole_world=False)

# sites

loading data for sites works very similarly to satellites

In [ ]:
met_datadir = "/group/chemistry/acrg/met_archive/NAME/EUROPE_met/EUROPE_Met_10magl_"
fp_datadir = "/group/chemistry/acrg/LPDM/fp_NAME_pre20210701/EUROPE/MHD-10magl_EUROPE_"

In [ ]:
mhd_sq = LoadSquareSiteData(year="2016", site="MHD", month="01", size=100, freq=10, verbose=True, fp_datadir=fp_datadir, met_args={"met_datadir":met_datadir})

In [ ]:
plt.imshow(mhd_sq.topog.landcover.sel(time=mhd_sq.topog.topog.time[0]), origin="lower")

In [ ]:
mhd_dom = LoadDomainSiteData(year="2016", site="MHD", month="01", freq=10, verbose=True, fp_datadir=fp_datadir, met_args={"met_datadir":met_datadir})

In [ ]:
plt.imshow(mhd_dom.topog.landcover, origin="lower")

In [ ]:
mhd_dom.met

can also get the inputs using the same function - note that at the moment, these regions have met only at 10magl (ie at a single level) and therefore should be treated like surface variables

In [ ]:
inputs, names = get_square_satellite_inputs(mhd_dom, {"Wind_Speed":[], "Temperature":[]}, static_variables=["lat_coords", "topog", "landcover", "landcover_disaggregated"], time_deltas=[100], verbose=True, return_asarray=False, return_variable_names=True)